# IP 01 — Operational Input Contract

Document what must be available before an operational prediction can be made.

In [1]:
# Import libraries
from pathlib import Path
import sys
import json
import yaml
import pandas as pd
import matplotlib.pyplot as plt

In [2]:
# Define config paths
PROJECT_ROOT = Path.cwd()
while (
    not (PROJECT_ROOT / "src").exists()
    and PROJECT_ROOT != PROJECT_ROOT.parent
):
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

CONFIG_PATH = (PROJECT_ROOT / "configs" / "integrated_pipeline.yaml")

with CONFIG_PATH.open("r", encoding="utf-8") as handle:
    CONFIG = yaml.safe_load(handle)

CONFIG_PATH

WindowsPath('e:/jcuenca/OneDrive - GUSCanada/5toTerm/01_Capstone/DataLocal/ontario-electricity-peak-risk/configs/integrated_pipeline.yaml')

Operational inputs must ultimately provide:
- forecast origin;
- requested FSA(s);
- recent historical electricity demand needed for lag/rolling features;
- forecast-available weather for the next 24 hours;
- deterministic calendar variables;
- any other frozen model features.

The integrated model must not use future observed demand or future observed weather as if they were available at prediction time.

In [3]:
ARTIFACTS = PROJECT_ROOT / CONFIG["paths"]["artifacts_dir"]

checks = []

for horizon in range(1, 25):
    checks.append(
        {
            "task": "Random Forest",
            "horizon": horizon,
            "path": str(
                ARTIFACTS
                / "forecasting_random_forest"
                / f"rf_h{horizon:02d}.joblib"
            ),
            "exists": (
                ARTIFACTS
                / "forecasting_random_forest"
                / f"rf_h{horizon:02d}.joblib"
            ).exists(),
        }
    )

    checks.append(
        {
            "task": "XGBoost",
            "horizon": horizon,
            "path": str(
                ARTIFACTS
                / "peak_risk_xgboost"
                / f"xgb_h{horizon:02d}.joblib"
            ),
            "exists": (
                ARTIFACTS
                / "peak_risk_xgboost"
                / f"xgb_h{horizon:02d}.joblib"
            ).exists(),
        }
    )

checks = pd.DataFrame(checks)

display(checks)
print("All 48 model artifacts available:", checks["exists"].all())

rf_metadata_path = (
    ARTIFACTS
    / "forecasting_random_forest"
    / "metadata.json"
)

xgb_metadata_path = (
    ARTIFACTS
    / "peak_risk_xgboost"
    / "metadata.json"
)

peak_threshold_path = (
    ARTIFACTS
    / "peak_risk_xgboost"
    / "peak_thresholds.parquet"
)

print("RF metadata:", rf_metadata_path.exists())
print("XGB metadata:", xgb_metadata_path.exists())
print("Peak thresholds:", peak_threshold_path.exists())


,task,horizon,path,exists
0,Random Forest,1,e:\jcuenca\OneDrive - GUSCanada\5toTerm\01_Cap...,True
1,XGBoost,1,e:\jcuenca\OneDrive - GUSCanada\5toTerm\01_Cap...,True
2,Random Forest,2,e:\jcuenca\OneDrive - GUSCanada\5toTerm\01_Cap...,True
3,XGBoost,2,e:\jcuenca\OneDrive - GUSCanada\5toTerm\01_Cap...,True
4,Random Forest,3,e:\jcuenca\OneDrive - GUSCanada\5toTerm\01_Cap...,True
5,XGBoost,3,e:\jcuenca\OneDrive - GUSCanada\5toTerm\01_Cap...,True
6,Random Forest,4,e:\jcuenca\OneDrive - GUSCanada\5toTerm\01_Cap...,True
7,XGBoost,4,e:\jcuenca\OneDrive - GUSCanada\5toTerm\01_Cap...,True
8,Random Forest,5,e:\jcuenca\OneDrive - GUSCanada\5toTerm\01_Cap...,True
9,XGBoost,5,e:\jcuenca\OneDrive - GUSCanada\5toTerm\01_Cap...,True


All 48 model artifacts available: True
RF metadata: True
XGB metadata: True
Peak thresholds: True


In [4]:

rf_metadata = json.loads(
    rf_metadata_path.read_text(encoding="utf-8")
)

xgb_metadata = json.loads(
    xgb_metadata_path.read_text(encoding="utf-8")
)

input_contract = pd.DataFrame(
    [
        {
            "input": "forecast_origin",
            "required": True,
            "description": "Timestamp from which h+1...h+24 are predicted.",
        },
        {
            "input": "FSA",
            "required": True,
            "description": "One or more supported FSA zones.",
        },
        {
            "input": "recent demand history",
            "required": True,
            "description": "Required to construct demand lags and rolling features.",
        },
        {
            "input": "24-hour weather forecast",
            "required": True,
            "description": "Forecast-available weather for target hours.",
        },
        {
            "input": "calendar information",
            "required": True,
            "description": "Hour/day/month/season features derived from target timestamp.",
        },
    ]
)

display(input_contract)


,input,required,description
0,forecast_origin,True,Timestamp from which h+1...h+24 are predicted.
1,FSA,True,One or more supported FSA zones.
2,recent demand history,True,Required to construct demand lags and rolling ...
3,24-hour weather forecast,True,Forecast-available weather for target hours.
4,calendar information,True,Hour/day/month/season features derived from ta...
